# 1. Data Cleaning

### Importing the data

In [1]:
import pandas as pd

data_path = "../data/raw_data/billing.csv"
df = pd.read_csv(data_path)
df.head()

,user_id,month,plan_tier,active_seats,mrr,discount_applied,invoices_overdue,support_ticket_count
0,f94d1824-8742-4000-8b6d-39d70958490b,2024-05,free,2,0.0,0,0,0
1,f94d1824-8742-4000-8b6d-39d70958490b,2024-06,free,2,0.0,0,0,0
2,f94d1824-8742-4000-8b6d-39d70958490b,2024-07,free,1,0.0,0,0,0
3,f94d1824-8742-4000-8b6d-39d70958490b,2024-08,free,1,0.0,0,0,0
4,f94d1824-8742-4000-8b6d-39d70958490b,2024-09,free,1,0.0,0,0,0


Some common errors or "messy-ness" in the data include:
1. Missing values
2. Duplicates
3. Inconsistent formatting
4. Outliers and typos
5. Data types

### Identifying missing values

In [2]:
df.isnull()

,user_id,month,plan_tier,active_seats,mrr,discount_applied,invoices_overdue,support_ticket_count
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
1000171,False,False,False,False,False,False,False,False
1000172,False,False,False,False,False,False,False,False
1000173,False,False,False,False,False,False,False,False
1000174,False,False,False,False,False,False,False,False


In [3]:
df.isnull().sum()

user_id                 0
month                   0
plan_tier               0
active_seats            0
mrr                     0
discount_applied        0
invoices_overdue        0
support_ticket_count    0
dtype: int64

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000176 entries, 0 to 1000175
Data columns (total 8 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   user_id               1000176 non-null  str    
 1   month                 1000176 non-null  str    
 2   plan_tier             1000176 non-null  str    
 3   active_seats          1000176 non-null  int64  
 4   mrr                   1000176 non-null  float64
 5   discount_applied      1000176 non-null  int64  
 6   invoices_overdue      1000176 non-null  int64  
 7   support_ticket_count  1000176 non-null  int64  
dtypes: float64(1), int64(4), str(3)
memory usage: 61.0 MB


Luckily here, we don't seem to have any missing values in this dataset to worry about :)

Let's take a look at a case that does have missing values so we know how to deal with them

In [5]:
sessions_data = "../data/raw_data/sessions.csv"
df1 = pd.read_csv(sessions_data)
df1.isnull().sum()


session_id               0
user_id                  0
session_start            0
session_end              0
device                   0
os                       0
app_version           4935
country                  0
session_length_sec       0
dtype: int64

In [24]:
df1[df1["app_version"].isnull()].head(5)

,session_id,user_id,session_start,session_end,device,os,app_version,country,session_length_sec
94,8aadbe61-862e-47f6-9362-458274e5da6d,a44b8970-3fec-47df-b07e-467e7ae8349f,2024-04-08 13:09:02.000000000,2024-04-08 13:26:34.590570108,mobile,ios,NaN,FR,1052
108,f03c9099-fc3f-4329-9f94-2feaab8cefa6,a44b8970-3fec-47df-b07e-467e7ae8349f,2025-05-06 02:05:26.882126616,2025-05-06 02:12:04.927632008,web,win,NaN,DE,398
127,77f46b54-3296-4bf8-bb0b-b6fcc963dfb6,55d94d74-fc7d-45d5-aab5-eefc4685c668,2024-11-09 16:09:51.548180558,2024-11-09 16:18:45.200497305,web,mac,NaN,CA,533
176,e6de1a1f-30bf-441e-9929-33f113445980,2e56c33e-4e00-41dc-9581-e81b2b5c00a2,2025-02-28 21:24:58.278385571,2025-02-28 21:49:47.352782515,web,win,NaN,US,1489
186,813fcc79-1556-4859-adf7-4f0dca8a6f43,9eeced56-d7f6-42cc-b26f-41ceec543a0b,2024-06-29 07:05:59.908507455,2024-06-29 07:10:22.034387719,mobile,ios,NaN,US,262


As a general rule of thumb, if there are not too many data inputs that have a NaN value, we normally just ignore and drop them

In [30]:
df2 = df1.dropna().reset_index(drop=True)
df2.isnull().sum()

session_id            0
user_id               0
session_start         0
session_end           0
device                0
os                    0
app_version           0
country               0
session_length_sec    0
dtype: int64

However, by doing so we lose the null data points

In [35]:
print(f"Data points before dropping: {len(df1)} \nData points after dropping: {len(df2)}")

Data points before dropping: 336717 
Data points after dropping: 331782


However, if the data is important, or there are too many NaN inputs, we need to "approximate" what the value would have been - this is called imputation 

Simple imputation methods are to just take the average/median of the column values that already exist - low complexity, however can lead to less accurate results

More complicated imputation methods include:
1. K-Nearest Neighbours (KNN) Imputation 
Instead of averaging the entire column, we find the rows that are the most similar to the missing values. We then average this subset of data, which helps preserve local patterns that exist in the data.

In [38]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=2)
X_imputed = imputer.fit_transform(df1)
X_imputed

ValueError: could not convert string to float: 'e1c0b7e9-a987-43d6-8290-dc88c2cdf5c0'

2. Multivariate Imputation by Chained Equations (MICE)
3. Using ML models for imputation

In [16]:
df2.isnull().sum()

user_id                    0
signup_date                0
plan_tier                  0
company_size               0
region                 38224
industry                   0
acquisition_channel        0
is_enterprise              0
churned_30d                0
churned_90d                0
downgraded                 0
expansion_event            0
dtype: int64